# Phase 3b — K-stress addendum (A100-40GB, High-RAM **OFF**)

Phase 2's K marginal was ~flat (0.94–1.03x) — a real null for its regime: KV demand
peaked at ~18% of the pool, so K's capacity channel never engaged. This notebook creates
the pressure deliberately and runs on the **40GB** A100 (High-RAM OFF — the toggle pins
the variant, PREREQ_RESULTS Check 1), because on 40GB **both** ceilings fit inside a
small grid:

- pool ≈ 0.85×40GB − ~16GB weights − overhead ≈ **~120–135k FP16-KV tokens**
- per-request context ≈ 7.4k-token unique document + 256 output ≈ 7.7k tokens
- → **FP16-KV ceiling ≈ 16 concurrent requests; FP8-KV ≈ 32**

Grid: {FP16-KV, FP8-KV} × concurrency **{8, 16, 32, 48}** × 2 repeats = 16 cells,
2 server launches. Each level has a job: 8 = below both ceilings (the Phase-2 null must
reproduce), 16 = ON the FP16 knee, 32 = FP16 saturated / ON the FP8 knee (max
divergence), 48 = beyond both — FP8's own plateau at ~2× FP16's, the demonstration the
80GB design could never fit in a sane concurrency range.

**Budget:** ~1.5–2 h ≈ **~20 units** (worst-case; thrashing cells are slow by design).
Independent of Sessions A–C — run it on either account, any time.

In [ ]:
# 1. Verify the 40GB card (High-RAM OFF)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
mem = int(subprocess.run(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().splitlines()[0])
assert mem < 50000, ("Got the 80GB A100 (%d MiB). Toggle High-RAM OFF and reconnect: "
                     "on 80GB the FP16 ceiling (~47) escapes this notebook grid.") % mem
units_before = input("Compute-units balance right now: ")

In [ ]:
# 2. Repo + Spec-Bench (long documents are built from its RAG passages)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK" 

In [ ]:
# 3. Isolated vLLM environment (PREREQ_RESULTS Check 6 recipe; ~6-8 min)
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)" 

In [ ]:
# 4. HF auth
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# 5. Harness self-check (~1 min, no GPU)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

In [ ]:
# 6. Sanity: the two server commands (identical except --kv-cache-dtype fp8)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/k_stress/kstress_*.yaml" --results-dir results --dry-run 2>&1 | grep "server command" 

In [ ]:
# 7. THE SWEEP: 16 cells, 2 launches, resumable. Cells at conc 32/48 under
# FP16-KV are SLOW BY DESIGN (preemption thrash) -- that's the phenomenon,
# not a hang; watch results/server_logs/ if unsure.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/k_stress/kstress_*.yaml" --results-dir results

In [ ]:
# 8. Pool size from the FP16-KV server log -> predicted-vs-measured plateau
!grep -h "GPU KV cache size" results/server_logs/*.log | tail -2
POOL = input("FP16-KV pool size in tokens (from the FP16 group line above): ").replace(",", "")
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.k_stress results --pool-tokens $POOL

In [ ]:
# 9. Preserve everything
units_after = input("Compute-units balance now: ")
print("Units burned:", units_before, "->", units_after)
!zip -qr phase3b_kstress_results.zip results
from google.colab import files
files.download("phase3b_kstress_results.zip")

## Reading the result

The four-row table should tell a two-knee story:

- **conc 8:** ratio ≈ 0.95–1.05, kv-usage well below 1.0, zero preemptions on both —
  the Phase-2 null reproducing below the ceiling. This row is what makes the rest
  credible.
- **conc 16:** FP16 kv-usage max ≈ 1.0 for the first time (batch pinned at the pool
  limit), FP8 cruising at ~0.5. Goodput ratio starts moving.
- **conc 32:** maximum divergence — FP16 batch flat at ~16 with preemptions > 0 and TTFT
  p95 blowing up; FP8 batch ~32, hitting ITS knee (kv-usage → 1.0).
- **conc 48:** both plateaued, FP8 plateau ≈ 2× FP16 — "FP8-KV doubles sustainable
  concurrency" measured on both sides, not extrapolated.

Cross-check the predicted plateau line (pool ÷ measured mean context) against the
measured batch columns; agreement within ~10% closes the loop between the capacity
arithmetic and the observed behavior. Decision-guide sentence this feeds: *enable FP8-KV
when projected demand (concurrency × context × 128 KiB) approaches the pool; below that
it costs ~5% on A100 (emulation tax) and buys nothing.*